# File Executor

本文档演示如何在 Runtime 容器中进行文件操作。 Nexus 提供了两种文件操作方式：
1. **上传文件**：使用 `upload_file` 将本地文件上传到 Runtime 容器
2. **下载文件**：使用 `download_file` 从 Runtime 容器下载文件到本地

## 初始化环境

首先导入必要的模块并配置日志。

In [1]:
from nexus.utils.log import setup_logging
from nexus.runtime import DockerRuntimeProvider

setup_logging()

## 创建并获取 Runtime 实例

创建 Docker Runtime Provider 并获取一个 Ubuntu 24.04 的运行时实例。

In [2]:
rp = DockerRuntimeProvider()
runtime = await rp.acquire(name='main', image='ubuntu:24.04')

2026-01-13 02:52:37 | INFO | nexus.runtime.docker | Image nexus-runtime:0.8.6 already exists, skipping build
2026-01-13 02:52:37 | INFO | nexus.runtime.docker | Runtime container nexus-runtime-0-8-6 already exists
2026-01-13 02:52:37 | INFO | nexus.runtime.docker | Starting container nexus-user-main...
2026-01-13 02:52:41 | INFO | nexus.runtime.docker | Runtime main ready at http://172.17.0.1:33158


## 上传文件（upload_file）

**方法签名**

```python
async def upload_file(
    files: list[FileItem],
    *,
    overwrite: bool = True,
) -> UploadFileResponse
```

**参数说明**

- `files` (`list[FileItem]`)：要上传的文件列表，每个 `FileItem` 包含：
  - `path` (`str`)：文件的绝对路径或相对路径（相对于 `NEXUS_WORKDIR` 环境变量或当前工作目录）
  - `content` (`str`)：文件内容（纯文本或 base64 编码）
  - `encoding` (`Literal['utf8', 'base64']`)：编码方式，默认 `utf8`
- `overwrite` (`bool`)：是否覆盖已存在的文件，默认 `True`

**返回值**

`UploadFileResponse` 包含：
- `results` (`list[FileOperationResult]`)：每个文件的操作结果
  - `path` (`str`)：文件路径
  - `success` (`bool`)：操作是否成功
  - `message` (`str | None`)：成功或错误信息

**实现位置**

- Executor 实现：`nexus/extensions/executors/file.py:93-127`
- Mixin 方法：`nexus/runtime/executor_mixin.py:113-132`

### 上传纯文本文件

In [3]:
from nexus.extensions.executors.file import FileItem

# 上传纯文本文件
result = await runtime.upload_file(
    files=[
        FileItem(
            path='/tmp/hello.txt',
            content='Hello, Nexus!',
            encoding='utf8'
        )
    ]
)

for file_result in result.results:
    print(f"Path: {file_result.path}")
    print(f"Success: {file_result.success}")
    print(f"Message: {file_result.message}")

2026-01-13 02:52:41 | INFO | nexus.core.registry | Imported 'nexus.extensions.agents.demo.tools' with 2 registrations:
2026-01-13 02:52:41 | INFO | nexus.core.registry |   - [tool] demo/think, demo/bash
2026-01-13 02:52:41 | INFO | nexus.core.registry | Imported 'nexus.extensions.agents.swe_agent.tools' with 32 registrations:
2026-01-13 02:52:41 | INFO | nexus.core.registry |   - [executor] create_browser_session, destroy_browser_session, browser_goto, browser_go_back, browser_go_forward, browser_click, browser_fill, browser_scroll, browser_press_key, browser_hover, browse [... (truncated log)] l_session, run_command_in_terminal_session, start_command_in_terminal_session, query_terminal_command_status, send_keys_to_terminal_session, capture_terminal_pane, terminate_terminal_session_processes
2026-01-13 02:52:41 | INFO | nexus.core.registry |   - [tool] swe_agent/bash, swe_agent/str_replace_editor, swe_agent/review_submit
2026-01-13 02:52:42 | INFO | nexus.core.registry | Imported 'nexu

### 验证文件已上传

In [4]:
# 验证文件内容
verify_result = await runtime.run_command('cat /tmp/hello.txt')
print(f"File content: {verify_result.stdout}")

File content: Hello, Nexus!


### 上传二进制文件（Base64 编码）

In [5]:
import base64

# 创建一个简单的二进制内容（模拟图片）
binary_content = b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01'
base64_content = base64.b64encode(binary_content).decode('utf-8')

result = await runtime.upload_file(
    files=[
        FileItem(
            path='/tmp/test.bin',
            content=base64_content,
            encoding='base64'
        )
    ]
)

print(f"Upload success: {result.results[0].success}")

Upload success: True


### 批量上传多个文件

In [6]:
# 批量上传
result = await runtime.upload_file(
    files=[
        FileItem(path='/tmp/file1.txt', content='Content 1', encoding='utf8'),
        FileItem(path='/tmp/file2.txt', content='Content 2', encoding='utf8'),
        FileItem(path='/tmp/subdir/file3.txt', content='Content 3', encoding='utf8'),
    ],
    overwrite=True
)

for file_result in result.results:
    print(f"{file_result.path}: {'✓' if file_result.success else '✗'}")

/tmp/file1.txt: ✓
/tmp/file2.txt: ✓
/tmp/subdir/file3.txt: ✓


## 下载文件（download_file）

**方法签名**

```python
async def download_file(
    files: list[FileItem],
) -> DownloadFileResponse
```

**参数说明**

- `files` (`list[FileItem]`)：要下载的文件列表，每个 `FileItem` 只需指定：
  - `path` (`str`)：文件路径
  - `encoding` (`Literal['utf8', 'base64']`)：期望的编码方式

**返回值**

`DownloadFileResponse` 包含：
- `files` (`list[FileItem]`)：成功下载的文件，包含 `path`、`content` 和 `encoding`
- `errors` (`list[FileOperationResult] | None`)：失败的文件操作（如果有）

**实现位置**

- Executor 实现：`nexus/extensions/executors/file.py:130-159`
- Mixin 方法：`nexus/runtime/executor_mixin.py:134-150`

### 下载文本文件

In [7]:
# 下载文本文件
result = await runtime.download_file(
    files=[
        FileItem(path='/tmp/hello.txt', encoding='utf8')
    ]
)

if result.files:
    file = result.files[0]
    print(f"Downloaded: {file.path}")
    print(f"Content: {file.content}")
    print(f"Encoding: {file.encoding}")

Downloaded: /tmp/hello.txt
Content: Hello, Nexus!
Encoding: utf8


### 下载二进制文件

In [8]:
# 下载二进制文件
result = await runtime.download_file(
    files=[
        FileItem(path='/tmp/test.bin', encoding='base64')
    ]
)

if result.files:
    file = result.files[0]
    # 解码 base64 内容
    binary_data = base64.b64decode(file.content)
    print(f"Downloaded binary file: {file.path}")
    print(f"Binary data length: {len(binary_data)} bytes")
    print(f"First 20 bytes: {binary_data[:20]}")

Downloaded binary file: /tmp/test.bin
Binary data length: 20 bytes
First 20 bytes: b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01'


### 批量下载多个文件

In [9]:
# 批量下载
result = await runtime.download_file(
    files=[
        FileItem(path='/tmp/file1.txt', encoding='utf8'),
        FileItem(path='/tmp/file2.txt', encoding='utf8'),
        FileItem(path='/tmp/nonexistent.txt', encoding='utf8'),  # 不存在的文件
    ]
)

print(f"Successfully downloaded {len(result.files)} files:")
for file in result.files:
    print(f"  - {file.path}")

if result.errors:
    print(f"\nFailed to download {len(result.errors)} files:")
    for error in result.errors:
        print(f"  - {error.path}: {error.message}")

Successfully downloaded 2 files:
  - /tmp/file1.txt
  - /tmp/file2.txt

Failed to download 1 files:
  - /tmp/nonexistent.txt: File does not exist


## 完整工作流示例：上传-修改-下载

In [10]:
# 1. 上传原始文件
await runtime.upload_file(
    files=[FileItem(path='/tmp/workflow.txt', content='Original content', encoding='utf8')]
)

# 2. 在 Runtime 中修改文件
await runtime.run_command('echo "\nAppended line" >> /tmp/workflow.txt')

# 3. 下载修改后的文件
result = await runtime.download_file(
    files=[FileItem(path='/tmp/workflow.txt', encoding='utf8')]
)

print("Modified file content:")
print(result.files[0].content)

Modified file content:
Original content
Appended line



## 实现位置

- File Executor 实现：`nexus/extensions/executors/file.py`
- Mixin 方法：`nexus/runtime/executor_mixin.py:111-150`

## 清理资源

使用完毕后释放所有 Runtime 实例。

In [11]:
await rp.release_all()

2026-01-13 02:52:42 | INFO | nexus.runtime.docker | Stopping container nexus-user-main...
2026-01-13 02:52:42 | INFO | nexus.runtime.docker | Runtime main released
